# Does activation steering move the model's internal state?

`scripts/shell/capture_steered_states.sh` runs the judge (no-value prompt) on comments three times — unsteered, and with a
value's steering vector added at the steered layers at +alpha and -alpha — and saves the internal state of the prompt's
last token after each captured layer (`states.npz`: `ids`, and `<setting>_<layer>` arrays of shape (comments, hidden)).

This notebook scores each state along the steering vector (how far it points in the value's direction) and compares the
settings, one layer at a time and across layers. Set the run below.

In [18]:
LAYER = 17                                                                              # for the single-layer plot
RUN = "../data/experiments/civil-comments/05_activation_steering/steered_states_toxicity_boundary"
SAMPLE = "../data/processed/civil-comments/boundary_n100_train/sample.csv"

In [19]:
import json
import textwrap

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde

meta = json.load(open(f"{RUN}/meta.json"))
z = np.load(f"{RUN}/states.npz")
VALUE, STEERED, CAPTURED = meta["value"], meta["steered_layers"], meta["captured_layers"]
vectors = np.load(f"../{meta['vectors']}/{VALUE}.npz")
settings = list(meta["settings"])                      # e.g. unsteered, toxicity@+68.6, not_toxicity@-68.6
COLORS = {s: ("#555555" if a == 0 else "#2e6fac" if a > 0 else "#d1495b") for s, a in meta["settings"].items()}
print(f"{meta['model']}: {meta['n']} comments, value {VALUE}, steered layers {STEERED[0]}-{STEERED[-1]}, "
      f"captured {CAPTURED[0]}-{CAPTURED[-1]}; settings {settings}")

openai/gpt-oss-20b: 100 comments, value toxicity, steered layers 7-19, captured 0-23; settings ['unsteered', 'toxicity@+68.6', 'not_toxicity@-68.6']


## Scores

A state's score is its dot product with the layer's steering vector scaled to length 1. Layers without their own vector
use the nearest steered layer's (before the first, the first; after the last, the last).

In [20]:
def direction(layer):
    l = min(max(layer, STEERED[0]), STEERED[-1])
    v = vectors[f"layer_{l}"].astype(float)
    return v / np.linalg.norm(v)

def scores(setting, layer):
    """One score per comment: how far its state at `layer` points along the steering vector."""
    return z[f"{setting}_{layer}"].astype(float) @ direction(layer)

base = settings[0]
for s in settings[1:]:
    print(f"layer {LAYER}, {s}: average shift {(scores(s, LAYER) - scores(base, LAYER)).mean():+.1f} "
          f"(unsteered spread across comments: sd {scores(base, LAYER).std():.1f})")

layer 17, toxicity@+68.6: average shift +494.0 (unsteered spread across comments: sd 18.4)
layer 17, not_toxicity@-68.6: average shift -537.5 (unsteered spread across comments: sd 18.4)


## One layer

In [ ]:
def curves(layer):
    """Per setting: (x grid, smoothed density, scores) of the comments' scores at `layer`, on a shared grid."""
    sc = {s: scores(s, layer) for s in settings}
    lo, hi = min(v.min() for v in sc.values()), max(v.max() for v in sc.values())
    g = np.linspace(lo - 0.08 * (hi - lo) - 5, hi + 0.08 * (hi - lo) + 5, 400)
    return {s: (g, gaussian_kde(v)(g) if v.std() > 0 else np.zeros_like(g), v) for s, v in sc.items()}

def add_curves(fig, layer, row=None, col=None, legend=False):
    for s, (g, dens, v) in curves(layer).items():
        fig.add_trace(go.Scatter(x=g, y=dens, mode="lines", fill="tozeroy", line=dict(color=COLORS[s], width=2),
                                 opacity=0.8, name=s, legendgroup=s, showlegend=legend,
                                 hovertemplate=f"{s}<br>score %{{x:.0f}}<br>average {v.mean():.0f}<extra></extra>"),
                      row=row, col=col)
        fig.add_vline(x=v.mean(), line=dict(color=COLORS[s], width=1, dash="dot"), row=row, col=col)

fig = go.Figure()
add_curves(fig, LAYER, legend=True)
fig.update_layout(title=f"<b>Layer {LAYER}: scores along the {VALUE} steering vector</b> ({meta['n']} comments)",
                  xaxis_title=f"low  \u2190  {VALUE}  \u2192  high", yaxis=dict(showticklabels=False, title=None),
                  template="simple_white", height=420, width=1000, legend=dict(x=0, y=1))
fig.show()

## One layer, comment by comment

Each dot is one comment under one setting (rows; the small up-and-down scatter is random). Hover to read the comment, its
human toxicity score and consensus, and its score along the vector.

## Every layer

In [21]:
cols = 4
rows = -(-len(CAPTURED) // cols)
titles = [f"<b>layer {l}</b>" + (" (steered)" if l in STEERED else "") for l in CAPTURED]
fig = make_subplots(rows=rows, cols=cols, subplot_titles=titles, vertical_spacing=0.06, horizontal_spacing=0.04)
for i, l in enumerate(CAPTURED):
    add_curves(fig, l, row=i // cols + 1, col=i % cols + 1, legend=i == 0)
fig.update_yaxes(showticklabels=False)
for ann, l in zip(fig.layout.annotations, CAPTURED):
    ann.update(x=ann.x - 0.11, xanchor="left", font=dict(size=12, color="#4b3f8f" if l in STEERED else "#333"))
fig.update_layout(title=f"<b>Scores along the {VALUE} steering vector, every captured layer</b> (own x scale per panel)",
                  template="simple_white", height=230 * rows + 120, width=1200, legend=dict(orientation="h", y=1.04, x=1,
                  xanchor="right"))
fig.show()

## Shift by layer

In [ ]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                    subplot_titles=[f"<b>A. Average shift along the {VALUE} vector</b>", "<b>B. Total change in the state, % of its size</b>"])
spread = np.array([scores(base, l).std() for l in CAPTURED])
fig.add_trace(go.Scatter(x=CAPTURED + CAPTURED[::-1], y=list(2 * spread) + list(-2 * spread[::-1]), fill="toself",
                         fillcolor="rgba(150,150,150,0.3)", line=dict(width=0), hoverinfo="skip",
                         name="spread across comments (+/-2 sd)"), row=1, col=1)
for s in settings[1:]:
    shift = [(scores(s, l) - scores(base, l)).mean() for l in CAPTURED]
    change = [100 * np.linalg.norm(z[f"{s}_{l}"] - z[f"{base}_{l}"], axis=1).mean() / np.linalg.norm(z[f"{base}_{l}"], axis=1).mean()
              for l in CAPTURED]
    fig.add_trace(go.Scatter(x=CAPTURED, y=shift, mode="lines+markers", line=dict(color=COLORS[s], width=2.5), name=s,
                             legendgroup=s, hovertemplate="layer %{x}: shift %{y:+.0f}<extra>" + s + "</extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(x=CAPTURED, y=change, mode="lines+markers", line=dict(color=COLORS[s], width=2.5), name=s,
                             legendgroup=s, showlegend=False, hovertemplate="layer %{x}: %{y:.0f}%<extra>" + s + "</extra>"),
                  row=1, col=2)
for col in (1, 2):
    fig.add_vrect(x0=STEERED[0] - 0.5, x1=STEERED[-1] + 0.5, fillcolor="#e8e3f7", opacity=0.5, line_width=0, layer="below",
                  row=1, col=col)
    fig.update_xaxes(title_text="layer (shaded: steered)", row=1, col=col)
fig.update_yaxes(title_text="shift vs unsteered", row=1, col=1)
fig.update_yaxes(title_text="% of the state's size", rangemode="tozero", row=1, col=2)
for ann in fig.layout.annotations:
    ann.update(x=ann.x - 0.2, xanchor="left")
fig.update_layout(template="simple_white", height=450, width=1200, legend=dict(orientation="h", y=-0.2))
fig.show()